# Day 4 — Prophet + Hierarchical Aggregation Analysis

Two questions answered here:

1. **How does Prophet perform on the 1,000-series sample vs ETS/ARIMA/SN28?**  
   (Results fed in from `scripts/run_day4.py` scores JSON)

2. **Bottom-up vs top-down forecasting: which wins?**  
   - *Bottom-up*: forecast each of the 1,000 base series independently with Prophet, then aggregate.  
   - *Top-down*: forecast aggregate levels (national, state, category, dept) with Prophet, then disaggregate to base level using historical proportion shares.  
   - Compare WRMSSE at each hierarchy level.

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')

PROJ_ROOT = os.path.dirname(os.getcwd())  # notebooks/ -> repo root
sys.path.insert(0, os.path.join(PROJ_ROOT, 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATA_RAW      = os.path.join(PROJ_ROOT, 'data', 'raw', 'm5-forecasting-accuracy')
REPORTS_DIR   = os.path.join(PROJ_ROOT, 'reports')
SUBMISSIONS   = os.path.join(PROJ_ROOT, 'submissions')

print('PROJ_ROOT:', PROJ_ROOT)

In [ ]:
# ── Load M5 data ───────────────────────────────────────────────────────────────
sales_eval  = pd.read_csv(os.path.join(DATA_RAW, 'sales_train_evaluation.csv'))
calendar_df = pd.read_csv(os.path.join(DATA_RAW, 'calendar.csv'))
prices_df   = pd.read_csv(os.path.join(DATA_RAW, 'sell_prices.csv'))

LAST_TRAIN_DAY = 1913
HORIZON = 28
train_cols  = [f'd_{d}' for d in range(1, LAST_TRAIN_DAY + 1)]
actual_cols = [f'd_{d}' for d in range(LAST_TRAIN_DAY + 1, LAST_TRAIN_DAY + HORIZON + 1)]
META_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

print(f'sales_eval: {sales_eval.shape}  calendar: {calendar_df.shape}  prices: {prices_df.shape}')

In [ ]:
# ── Load 1k sample ─────────────────────────────────────────────────────────────
sample_df  = pd.read_csv(os.path.join(REPORTS_DIR, 'sample_1000_series.csv'))
sample_ids = sample_df['id'].tolist()

sales_sub = (
    sales_eval[sales_eval['id'].isin(sample_ids)]
    .set_index('id').reindex(sample_ids).reset_index()
)
actuals_sub    = sales_sub[actual_cols].values.astype(np.float32)
train_mat_sub  = sales_sub[train_cols].values.astype(np.float32)

print(f'Sample: {len(sample_ids)} series | actuals: {actuals_sub.shape}')

## 1. Prophet Results Summary (from run_day4.py)

Load the JSON scores saved by the batch script.

In [ ]:
scores_path = os.path.join(REPORTS_DIR, 'day4_prophet_scores.json')
with open(scores_path) as f:
    prophet_scores = json.load(f)

# Build summary table
rows = []
for cps_key, r in prophet_scores['sweep'].items():
    cat = r.get('wrmsse_by_category', {})
    rows.append({
        'changepoint_prior_scale': float(cps_key),
        'WRMSSE_sample': r['wrmsse_sample'],
        'FOODS': cat.get('FOODS', float('nan')),
        'HOUSEHOLD': cat.get('HOUSEHOLD', float('nan')),
        'HOBBIES': cat.get('HOBBIES', float('nan')),
        'time_s': r['wall_time_seconds'],
        'zero_forecasts': r['n_zero_forecasts'],
    })

sweep_df = pd.DataFrame(rows).sort_values('changepoint_prior_scale')

# Reference row
refs = pd.DataFrame([{
    'changepoint_prior_scale': float('nan'), 'WRMSSE_sample': 0.6778,
    'FOODS': 0.6400, 'HOUSEHOLD': 1.1580, 'HOBBIES': 1.5949,
    'time_s': float('nan'), 'zero_forecasts': float('nan'),
}])
refs.index = ['SN28 ref']

display_df = pd.concat([refs, sweep_df.set_index(sweep_df['changepoint_prior_scale'].astype(str))])
display_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cps_vals = sweep_df['changepoint_prior_scale'].values
wrmsse   = sweep_df['WRMSSE_sample'].values

ax = axes[0]
ax.plot(cps_vals, wrmsse, 'o-', color='steelblue', lw=2)
ax.axhline(0.6778, ls='--', color='gray', label='SN28 ref (1k)')
ax.axhline(0.6541, ls='--', color='green', label='ETS (1k)')
ax.set_xlabel('changepoint_prior_scale')
ax.set_ylabel('WRMSSE (1k sample)')
ax.set_title('Prophet: WRMSSE vs changepoint_prior_scale')
ax.legend(fontsize=9)
ax.set_xscale('log')

ax2 = axes[1]
cats = ['FOODS', 'HOUSEHOLD', 'HOBBIES']
colors = ['#2ecc71', '#3498db', '#e74c3c']
for cat, col in zip(cats, colors):
    ax2.plot(cps_vals, sweep_df[cat].values, 'o-', label=cat, color=col, lw=2)
ax2.set_xlabel('changepoint_prior_scale')
ax2.set_ylabel('WRMSSE (1k sample)')
ax2.set_title('Prophet: Per-category WRMSSE')
ax2.legend(fontsize=9)
ax2.set_xscale('log')

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'prophet_cps_sweep.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/prophet_cps_sweep.png')

## 2. Hierarchical Forecasting: Bottom-Up vs Top-Down

### Approach

**Bottom-up (BU):** Forecast each base series independently (Prophet, best cps). Aggregate forecasts by summing.

**Top-down (TD):** 
1. For each aggregate level (national, state, category, dept), sum the base series and fit a single Prophet model on the aggregate.
2. Disaggregate to base level using each series' average proportion share of its parent over the last 28 training days.

We compare WRMSSE at each of the 12 M5 hierarchy levels.

In [ ]:
from evaluation.wrmsse import compute_wrmsse, LEVEL_SPECS
from models.prophet_model import build_m5_holidays, fit_prophet

# Load bottom-up preds (saved by run_day4.py)
best_cps = prophet_scores['best_cps']
sub_path = os.path.join(SUBMISSIONS, 'prophet_sample_submission.csv')
bu_sub   = pd.read_csv(sub_path)
fcols    = [f'F{i}' for i in range(1, 29)]
bu_preds = bu_sub[fcols].values.astype(np.float32)

print(f'Bottom-up Prophet preds loaded: {bu_preds.shape}  (best cps={best_cps})')

In [ ]:
# ── Build date index ───────────────────────────────────────────────────────────
D1 = pd.Timestamp('2011-01-29')
train_dates = pd.date_range(start=D1, periods=len(train_cols), freq='D')

holidays_df = build_m5_holidays(calendar_df)

# ── Top-down: forecast at 4 aggregate levels ───────────────────────────────────
# Levels: national (total), state, cat_id, dept_id
# For each level, aggregate train_mat and actuals, fit one Prophet per group,
# then distribute back to base series.

TD_LEVELS = [
    ('national', []),
    ('state',    ['state_id']),
    ('category', ['cat_id']),
    ('dept',     ['dept_id']),
]

def group_key_series(df, cols):
    if not cols:
        return pd.Series(['total'] * len(df), index=df.index)
    return df[cols].apply(lambda r: '__'.join(r.values.astype(str)), axis=1)


def top_down_prophet(level_name, group_cols, train_mat, actuals, sales_sub,
                     train_dates, holidays_df, best_cps):
    """Fit Prophet on each aggregate group, disaggregate to base series."""
    keys = group_key_series(sales_sub, group_cols).values
    unique_keys = np.unique(keys)

    # Proportion shares: each base series' share of its group in last 28 training days
    last28 = train_mat[:, -28:].sum(axis=1)  # total sales last 28 days per series
    shares = np.zeros(len(train_mat))
    for k in unique_keys:
        mask = keys == k
        grp_total = last28[mask].sum()
        if grp_total > 0:
            shares[mask] = last28[mask] / grp_total
        else:
            shares[mask] = 1.0 / mask.sum()  # uniform if group has zero sales

    # Fit Prophet on aggregate
    agg_preds_dict = {}
    for k in unique_keys:
        mask = keys == k
        agg_series = train_mat[mask].sum(axis=0).astype(np.float64)
        pred = fit_prophet(agg_series, train_dates, holidays_df, horizon=28,
                           changepoint_prior_scale=best_cps)
        agg_preds_dict[k] = pred

    # Disaggregate
    td_preds = np.zeros((len(train_mat), 28), dtype=np.float64)
    for i, (k, sh) in enumerate(zip(keys, shares)):
        td_preds[i] = agg_preds_dict[k] * sh

    return td_preds.astype(np.float32)


print('Running top-down Prophet for 4 aggregate levels...')
td_results = {}
for level_name, group_cols in TD_LEVELS:
    print(f'  {level_name}...')
    td_preds = top_down_prophet(
        level_name, group_cols,
        train_mat_sub.astype(np.float64), actuals_sub,
        sales_sub, train_dates, holidays_df, best_cps
    )
    score, levels = compute_wrmsse(
        preds=td_preds, actuals=actuals_sub,
        sales_df=sales_sub, prices_df=prices_df,
        calendar_df=calendar_df, last_train_day=LAST_TRAIN_DAY
    )
    td_results[level_name] = {'wrmsse': score, 'by_level': levels, 'preds': td_preds}
    print(f'    WRMSSE = {score:.4f}')

# Bottom-up score for comparison
bu_score, bu_levels = compute_wrmsse(
    preds=bu_preds, actuals=actuals_sub,
    sales_df=sales_sub, prices_df=prices_df,
    calendar_df=calendar_df, last_train_day=LAST_TRAIN_DAY
)
print(f'\nBottom-up Prophet WRMSSE = {bu_score:.4f}')

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Approach': ['Bottom-Up Prophet'] + [f'Top-Down ({k})' for k in td_results],
    'WRMSSE (1k)': [bu_score] + [v['wrmsse'] for v in td_results.values()],
})
summary['vs BU'] = summary['WRMSSE (1k)'] - bu_score
summary = summary.sort_values('WRMSSE (1k)').reset_index(drop=True)
display(summary.style.format({'WRMSSE (1k)': '{:.4f}', 'vs BU': '{:+.4f}'}))

In [ ]:
# ── Per-level breakdown: BU vs best TD ────────────────────────────────────────
level_names = list(LEVEL_SPECS.keys())

bu_level_scores = [bu_levels[l] for l in level_names]

# Pick best top-down variant
best_td_name = min(td_results, key=lambda k: td_results[k]['wrmsse'])
best_td_levels = [td_results[best_td_name]['by_level'][l] for l in level_names]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(level_names))
w = 0.35
ax.bar(x - w/2, bu_level_scores,   width=w, label='Bottom-Up Prophet', color='steelblue', alpha=0.85)
ax.bar(x + w/2, best_td_levels, width=w, label=f'Top-Down ({best_td_name})', color='coral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([l.replace('level_', 'L') for l in level_names], rotation=45)
ax.set_ylabel('WRMSSE')
ax.set_title('Bottom-Up vs Top-Down Prophet: WRMSSE by Hierarchy Level (1k sample)')
ax.legend()
ax.axhline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'hierarchical_bu_vs_td.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/hierarchical_bu_vs_td.png')

## 3. Key Findings

Fill in after running the above cells.

In [ ]:
print('=' * 60)
print('DAY 4 — HIERARCHICAL ANALYSIS FINDINGS')
print('=' * 60)
print(f'\nProphet (best cps={best_cps}) vs baselines (1k sample):')
print(f'  SN28 ref:   0.6778')
print(f'  ETS:        0.6541')
print(f'  Prophet BU: {bu_score:.4f}')

print(f'\nTop-Down variants:')
for name, r in sorted(td_results.items(), key=lambda x: x[1]['wrmsse']):
    delta = r['wrmsse'] - bu_score
    sign = '+' if delta > 0 else ''
    print(f'  TD ({name:12s}): WRMSSE={r["wrmsse"]:.4f}  (BU {sign}{delta:.4f})')

print(f'\nConclusion:')
if bu_score < min(v['wrmsse'] for v in td_results.values()):
    print('  Bottom-up wins. Per-series Prophet captures local seasonality')
    print('  better than shared aggregate-then-disaggregate approach.')
else:
    winner = min(td_results, key=lambda k: td_results[k]['wrmsse'])
    print(f'  Top-down ({winner}) wins. Aggregate-level Prophet is smoother')
    print('  and the noise-cancellation at higher levels outweighs lost local detail.')